[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/transformer_translation.ipynb)

# A Transformer from scratch: translation

We build the **original Transformer** of the slides (encoder–decoder, Vaswani et al., 2017) from its pieces, train it on the toy English → Spanish task of the `attention_mechanism` notebook, and then **break it on purpose** to see why each piece is there:

1. The data (the same task as before).
2. The pieces: multi-head attention, feed-forward network, positional encoding.
3. Encoder block, decoder block, and the full model with its **masks**.
4. Training in parallel with **teacher forcing**; generation one token at a time.
5. Looking inside: **cross-attention is classic attention**; encoder self-attention; masked decoder self-attention.
6. Experiment 1: **no positional encoding**.
7. Experiment 2: **no causal mask**.

The mechanics of each piece are explained, with numbers, in the `transformer_self_attention` notebook.

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=3, sci_mode=False)
# A small model: the CPU is enough. Each training takes about 45-90 s.
device = torch.device("cpu")

## 1. The data

English noun phrases joined by *and*, translated to Spanish: the adjective moves after the noun (*the black cat* → *el gato negro*), the article depends on the gender of the noun that comes **later** (*the house* → **la** casa), and the adjective agrees with the noun (*rojo* / *roja*).

In [ ]:
NOUNS = {"house": ("casa", "f"), "box": ("caja", "f"), "ball": ("pelota", "f"),
         "cat": ("gato", "m"), "dog": ("perro", "m"), "car": ("coche", "m")}
ADJECTIVES = {"red": ("rojo", "roja"), "black": ("negro", "negra"), "white": ("blanco", "blanca"),
              "blue": ("azul", "azul"), "green": ("verde", "verde")}   # (masculine, feminine)

def make_phrase(rng):
    noun = rng.choice(list(NOUNS))
    es_noun, gender = NOUNS[noun]
    article = "la" if gender == "f" else "el"
    if rng.random() < 0.6:
        adj = rng.choice(list(ADJECTIVES))
        return ["the", adj, noun], [article, es_noun, ADJECTIVES[adj][gender == "f"]]
    return ["the", noun], [article, es_noun]

def make_pair(n_phrases, rng):
    src, tgt = [], []
    for i in range(n_phrases):
        if i > 0:
            src.append("and")
            tgt.append("y")
        s, t = make_phrase(rng)
        src += s
        tgt += t
    return src, tgt

SPECIAL = ["<pad>", "<sos>", "<eos>"]
PAD, SOS, EOS = 0, 1, 2
src_vocab = SPECIAL + ["the", "and"] + list(ADJECTIVES) + list(NOUNS)
tgt_vocab = SPECIAL + ["el", "la", "y"] + sorted({w for pair in ADJECTIVES.values() for w in pair}) \
            + [es for es, _ in NOUNS.values()]
src_index = {w: i for i, w in enumerate(src_vocab)}
tgt_index = {w: i for i, w in enumerate(tgt_vocab)}

def to_tensors(pairs):
    # (src, tgt) word lists -> two padded tensors (batch, length), with <eos> at the end
    Tx = max(len(s) for s, _ in pairs) + 1
    Ty = max(len(t) for _, t in pairs) + 1
    x = torch.full((len(pairs), Tx), PAD)
    y = torch.full((len(pairs), Ty), PAD)
    for i, (s, t) in enumerate(pairs):
        ids = [src_index[w] for w in s] + [EOS]
        x[i, :len(ids)] = torch.tensor(ids)
        ids = [tgt_index[w] for w in t] + [EOS]
        y[i, :len(ids)] = torch.tensor(ids)
    return x, y

rng = random.Random(0)
for n in [1, 2, 3]:
    src, tgt = make_pair(n, rng)
    print(" ".join(src), " -> ", " ".join(tgt))

## 2. The pieces

### Multi-head attention

The same function as in the slides, for a **batch** of sequences. The queries come from `x_q` and the keys and values from `x_kv`:

- **self-attention**: `x_q` and `x_kv` are the same sequence;
- **cross-attention**: `x_q` comes from the decoder and `x_kv` from the encoder.

The `mask` says which keys each query may see (True = allowed). The layer keeps its last attention weights in `self.weights`, so we can look at them later.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.h, self.d_k = h, d_model // h
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        # (B, n, d_model) -> (B, h, n, d_k)
        B, n, _ = x.shape
        return x.view(B, n, self.h, self.d_k).transpose(1, 2)

    def forward(self, x_q, x_kv, mask=None):
        Q = self.split_heads(self.W_q(x_q))                       # (B, h, n_q, d_k)
        K = self.split_heads(self.W_k(x_kv))                      # (B, h, n_kv, d_k)
        V = self.split_heads(self.W_v(x_kv))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)    # (B, h, n_q, n_kv)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        A = F.softmax(scores, dim=-1)
        self.weights = A.detach()
        Z = (A @ V).transpose(1, 2).flatten(2)                    # concat heads: (B, n_q, d_model)
        return self.W_o(Z)

### Feed-forward network and positional encoding

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2, \qquad \text{PE}(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_\text{model}}}\right), \quad \text{PE}(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)$$

In [ ]:
def feed_forward(d_model, d_ff):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

def positional_encoding(n_positions, d_model):
    pos = torch.arange(n_positions).unsqueeze(1)
    angle = pos / 10000 ** (torch.arange(0, d_model, 2) / d_model)
    PE = torch.zeros(n_positions, d_model)
    PE[:, 0::2] = torch.sin(angle)
    PE[:, 1::2] = torch.cos(angle)
    return PE

## 3. Encoder block, decoder block and the full model

Both blocks use **pre-norm** residual connections: $x \mapsto x + \text{Sublayer}(\text{LayerNorm}(x))$.

- **Encoder block**: self-attention over the whole source (in both directions) + FFN.
- **Decoder block**: **masked** self-attention over the target words so far + **cross-attention** to the encoder output + FFN.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, h)
        self.ffn = feed_forward(d_model, d_ff)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)

    def forward(self, x, src_mask):
        y = self.norm1(x)
        x = x + self.self_attention(y, y, src_mask)
        return x + self.ffn(self.norm2(x))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, h)
        self.cross_attention = MultiHeadAttention(d_model, h)
        self.ffn = feed_forward(d_model, d_ff)
        self.norm1, self.norm2, self.norm3 = (nn.LayerNorm(d_model) for _ in range(3))

    def forward(self, y, memory, tgt_mask, src_mask):
        z = self.norm1(y)
        y = y + self.self_attention(z, z, tgt_mask)                    # target words so far
        y = y + self.cross_attention(self.norm2(y), memory, src_mask)  # Q: decoder, K and V: encoder
        return y + self.ffn(self.norm3(y))

### The masks

Both are boolean tensors of shape `(B, 1, n_q, n_kv)`, broadcast over the heads:

- **padding mask**: no query may look at a `<pad>` key;
- **causal mask** (decoder self-attention only): position $i$ may only look at positions $j \le i$.

Two switches, `use_positions` and `causal`, let us remove the positional encoding or the causal mask for the experiments of sections 6 and 7.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_model=64, h=4, N=2, d_ff=256, use_positions=True, causal=True, max_len=100):
        super().__init__()
        self.d_model, self.use_positions, self.causal = d_model, use_positions, causal
        self.src_embedding = nn.Embedding(len(src_vocab), d_model)
        self.tgt_embedding = nn.Embedding(len(tgt_vocab), d_model)
        self.register_buffer("PE", positional_encoding(max_len, d_model))
        self.encoder = nn.ModuleList(EncoderBlock(d_model, h, d_ff) for _ in range(N))
        self.decoder = nn.ModuleList(DecoderBlock(d_model, h, d_ff) for _ in range(N))
        self.encoder_norm, self.decoder_norm = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, len(tgt_vocab))                  # vector -> one logit per word

    def embed(self, embedding, ids):
        x = embedding(ids) * math.sqrt(self.d_model)                    # scaling used in the paper
        return x + self.PE[:ids.size(1)] if self.use_positions else x

    def encode(self, x):
        src_mask = (x != PAD)[:, None, None, :]                         # (B, 1, 1, Tx)
        h = self.embed(self.src_embedding, x)
        for block in self.encoder:
            h = block(h, src_mask)
        return self.encoder_norm(h), src_mask

    def decode(self, y_in, memory, src_mask):
        n = y_in.size(1)
        tgt_mask = (y_in != PAD)[:, None, None, :]                      # (B, 1, 1, Ty)
        if self.causal:
            tgt_mask = tgt_mask & torch.tril(torch.ones(n, n, dtype=torch.bool))   # (B, 1, Ty, Ty)
        h = self.embed(self.tgt_embedding, y_in)
        for block in self.decoder:
            h = block(h, memory, tgt_mask, src_mask)
        return self.out(self.decoder_norm(h))                           # (B, Ty, vocab)

    def forward(self, x, y_in):
        memory, src_mask = self.encode(x)
        return self.decode(y_in, memory, src_mask)

model = Transformer()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training and generation

### Training: all positions in parallel

The whole target sentence is known. The decoder input is the target **shifted one position to the right**, starting with `<sos>` (**teacher forcing**), and at every position the model must predict the next word:

In [ ]:
x, y = to_tensors([make_pair(1, random.Random(3))])

def shift_right(y):
    return torch.cat([torch.full((y.size(0), 1), SOS), y[:, :-1]], dim=1)

y_in = shift_right(y)
print("decoder input:", [tgt_vocab[i] for i in y_in[0]])
print("target       :", [tgt_vocab[i] for i in y[0]])

Thanks to the causal mask, position $i$ only sees the decoder input up to $i$, so the $T_y$ predictions are computed in **one** pass and the loss is the cross-entropy summed over all positions.

Transformers are sensitive to the learning rate at the start of training: the usual recipe is a **warmup** (the learning rate grows linearly for the first steps) followed by a decay.

In [ ]:
MAX_PHRASES = 6

def train(model, steps=1500, batch_size=64, lr=3e-3, warmup=100, seed=0):
    rng = random.Random(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98))
    schedule = lambda step: min(1.0, (step + 1) / warmup) * max(0.05, 1 - step / steps)   # warmup, then linear decay
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
    start = time.time()
    for step in range(1, steps + 1):
        x, y = to_tensors([make_pair(rng.randint(1, MAX_PHRASES), rng) for _ in range(batch_size)])
        logits = model(x, shift_right(y))                         # all positions at once
        loss = F.cross_entropy(logits.reshape(-1, len(tgt_vocab)), y.reshape(-1), ignore_index=PAD)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        if step % 300 == 0:
            print(f"step {step:5d}  loss {loss.item():.4f}  lr {scheduler.get_last_lr()[0]:.4f}  ({time.time() - start:.0f} s)")
    return model

torch.manual_seed(0)
model = train(Transformer())

### Generation: one token at a time

The output is not known: start with `<sos>`, predict the next word, append it to the decoder input, and run the decoder again, until `<eos>`. (A real implementation keeps the keys and values of the previous positions in a **KV cache** instead of recomputing them; here we recompute everything, which is simpler.)

In [ ]:
@torch.no_grad()
def translate(model, x, max_len=40):
    model.eval()
    memory, src_mask = model.encode(x)                     # the encoder runs once
    y = torch.full((x.size(0), 1), SOS)
    for _ in range(max_len):
        logits = model.decode(y, memory, src_mask)         # the decoder runs once per new word
        next_word = logits[:, -1].argmax(dim=-1, keepdim=True)
        y = torch.cat([y, next_word], dim=1)
        if (y == EOS).any(dim=1).all():
            break
    model.train()
    return y[:, 1:]

def to_words(ids):
    words = [tgt_vocab[i] for i in ids.tolist()]
    return words[:words.index("<eos>")] if "<eos>" in words else words

def translate_sentence(model, sentence):
    x, _ = to_tensors([(sentence.split(), [])])
    return " ".join(to_words(translate(model, x)[0]))

for s in ["the black cat", "the blue house and the dog",
          "the red ball and the white car and the green box and the cat"]:
    print(f"{s}\n   -> {translate_sentence(model, s)}")

In [ ]:
def evaluate(model, n_phrases, n=200, seed=1000):
    rng = random.Random(seed + n_phrases)
    pairs = [make_pair(n_phrases, rng) for _ in range(n)]
    x, _ = to_tensors(pairs)
    out = translate(model, x)
    return np.mean([to_words(out[i]) == tgt for i, (_, tgt) in enumerate(pairs)])

def accuracy_table(models):
    return pd.DataFrame({name: [evaluate(m, k) for k in range(1, MAX_PHRASES + 1)] for name, m in models.items()},
                        index=pd.Index(range(1, MAX_PHRASES + 1), name="phrases"))

accuracy_table({"Transformer": model})

**Explain:** compare with the GRU models of the `attention_mechanism` notebook (exact sentence accuracy: about 0 without attention for 4 or more phrases, about 1 with attention). The Transformer has **no recurrence at all**: everything it knows about the order of the words comes from the positional encoding and the causal mask.

## 5. Looking inside

### Cross-attention is classic attention

In the decoder, the queries come from the target words and the keys and values from the encoder output: it is the classic attention of the previous lecture, with $h$ heads. Each head gives one **alignment** heatmap: rows = output words, columns = input words.

**Predict first:** for *the blue house* → *la casa azul*, where should *la*, *casa* and *azul* look?

In [ ]:
def heatmap(M, x_labels, y_labels, title="", ax=None):
    ax = ax or plt.gca()
    ax.imshow(M, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    if M.shape[0] * M.shape[1] <= 80:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if M[i, j] > 0.6 else "black")
    ax.set_xticks(range(len(x_labels)), x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(y_labels)), y_labels, fontsize=8)
    ax.set_title(title, fontsize=9)

@torch.no_grad()
def run_and_collect(model, sentence):
    # Translates, then runs the model once more on (source, translation) to collect all the attention weights.
    x, _ = to_tensors([(sentence.split(), [])])
    out = translate(model, x)
    words = to_words(out[0]) + ["<eos>"]
    y = torch.tensor([[tgt_index[w] for w in words]])
    model.eval()
    model(x, shift_right(y))
    model.train()
    src = sentence.split() + ["<eos>"]
    return src, words, ["<sos>"] + words[:-1]

def show_cross_attention(model, sentence, layer=-1):
    src, words, _ = run_and_collect(model, sentence)
    A = model.decoder[layer].cross_attention.weights[0]          # (h, Ty, Tx)
    h = A.size(0)
    fig, axes = plt.subplots(1, h + 1, figsize=(3.2 * (h + 1), 0.35 * len(words) + 1.8))
    for i in range(h):
        heatmap(A[i].numpy(), src, words, f"head {i + 1}", ax=axes[i])
    heatmap(A.mean(0).numpy(), src, words, "average of the heads", ax=axes[h])
    fig.suptitle(f"cross-attention, decoder block {layer % len(model.decoder) + 1}")
    plt.tight_layout()
    plt.show()

show_cross_attention(model, "the blue house")

**Explain:** different heads may follow different relations (the word to translate, the noun that decides the gender, ...), and some may look rather unfocused: what matters is the combination. Try `layer=0`, and the slides' other example below.

In [ ]:
show_cross_attention(model, "the black cat and the white house")

### Encoder self-attention and masked decoder self-attention

In the encoder, every source word looks at every source word, in both directions. In the decoder, the causal mask leaves only the lower triangle.

In [ ]:
def show_self_attention(model, sentence, layer=0):
    src, words, dec_in = run_and_collect(model, sentence)
    A_enc = model.encoder[layer].self_attention.weights[0]       # (h, Tx, Tx)
    A_dec = model.decoder[layer].self_attention.weights[0]       # (h, Ty, Ty)
    h = A_enc.size(0)
    fig, axes = plt.subplots(2, h, figsize=(3.2 * h, 6.5))
    for i in range(h):
        heatmap(A_enc[i].numpy(), src, src, f"encoder, head {i + 1}", ax=axes[0, i])
        heatmap(A_dec[i].numpy(), dec_in, dec_in, f"decoder (masked), head {i + 1}", ax=axes[1, i])
    fig.suptitle(f"self-attention, block {layer + 1}")
    plt.tight_layout()
    plt.show()

show_self_attention(model, "the black cat and the white house")

**Explain:**

- In the encoder, does *the* look at its noun (it needs the gender)? Does the adjective look at its noun?
- In the decoder, why is everything above the diagonal exactly 0? Which position always has a single choice?

## 6. Experiment 1: no positional encoding

Self-attention is **permutation equivariant**: without positions, the encoder sees the source as a *bag* of words. *the black cat and the white dog* and *the white cat and the black dog* have exactly the same words.

**Predict first:** what will the accuracy be without positional encoding? Which sentences can it still translate?

In [ ]:
torch.manual_seed(0)
model_no_pos = train(Transformer(use_positions=False))

In [ ]:
print(accuracy_table({"Transformer": model, "no positional encoding": model_no_pos}))

for s in ["the black cat and the white dog", "the white cat and the black dog"]:
    print(f"\n{s}\n   with positions   : {translate_sentence(model, s)}\n   without positions: {translate_sentence(model_no_pos, s)}")

# The encoder output for the same word is identical in both sentences (up to rounding)
with torch.no_grad():
    m1, _ = model_no_pos.encode(to_tensors([("the black cat and the white dog".split(), [])])[0])
    m2, _ = model_no_pos.encode(to_tensors([("the white cat and the black dog".split(), [])])[0])
print("\nencoder output of 'cat' identical in both sentences?", torch.allclose(m1[0, 2], m2[0, 2], atol=1e-5))

**Explain:** with one phrase the bag of words is almost enough (*the*, one adjective, one noun). With two or more, the model cannot know which adjective goes with which noun, nor which noun comes first. Note that the decoder still knows *its own* positions in a way: the causal mask lets position $i$ see exactly $i$ words.

## 7. Experiment 2: no causal mask

Exercise 2 of the slides asks: *what would go wrong in training without the mask?*

**Predict first:** without the mask, will the training loss be higher or lower? And the translations?

In [ ]:
torch.manual_seed(0)
model_no_mask = train(Transformer(causal=False))

In [ ]:
print(accuracy_table({"Transformer": model, "no causal mask": model_no_mask}))
for s in ["the black cat", "the blue house and the dog"]:
    print(f"\n{s}\n   with mask   : {translate_sentence(model, s)}\n   without mask: {translate_sentence(model_no_mask, s)}")

# The training situation: teacher forcing with the CORRECT translation as decoder input
source, target = "the black cat and the white house", "el gato negro y la casa blanca"
x, y = to_tensors([(source.split(), target.split())])
with torch.no_grad():
    model_no_mask(x, shift_right(y))
dec_in = [tgt_vocab[i] for i in shift_right(y)[0]]
A = model_no_mask.decoder[1].self_attention.weights[0]          # block 2, (h, Ty, Ty)
fig, axes = plt.subplots(1, A.size(0), figsize=(3.2 * A.size(0), 3.6))
for i in range(A.size(0)):
    heatmap(A[i].numpy(), dec_in, dec_in, f"head {i + 1}", ax=axes[i])
fig.suptitle("decoder self-attention WITHOUT mask, block 2, during training (teacher forcing)")
plt.tight_layout()
plt.show()

**Explain:** during training, position $i$ of the decoder input contains the word that position $i-1$ must predict. Without the mask, each position simply **copies the next input word**: the loss goes down quickly, but the model learned nothing useful. In generation the future words do not exist yet, so there is nothing to copy. Look at the heatmaps: the row of each word puts much of its weight on the **next** column, the word it has to predict.

## Summary

- The original Transformer = an encoder (self-attention + FFN, stacked) and a decoder (masked self-attention + cross-attention + FFN, stacked), with residual connections and layer normalisation.
- Training processes all target positions in **parallel** with teacher forcing; generation produces **one token at a time**.
- Cross-attention is the classic attention of the previous lecture, with several heads.
- Without **positional encoding**, the encoder sees a bag of words; without the **causal mask**, the decoder learns to copy the answer and fails at generation.

## Your turn

1. **Weight tying**: make `self.out` use the transposed target embedding matrix (`self.out.weight = self.tgt_embedding.weight`). Does it still learn? How many parameters do you save?
2. **Learned positions**: replace the sinusoidal encoding by `nn.Embedding(max_len, d_model)`. Compare the accuracy and the training loss.
3. **Longer sentences** than in training: `evaluate(model, 7)` and `evaluate(model, 8)`. What happens, and why might the positional encoding be part of the problem?
4. **Size**: try `h=1`, `N=1` or `d_model=32`. What is the smallest model that still gets almost everything right?
5. **No warmup**: train with `warmup=1` and compare the loss curves.
6. **KV cache**: in `translate`, the decoder recomputes the keys and values of all previous positions at every step. How many times is the key of the first word computed for a 20-word translation? How could you avoid it?

In [ ]:
# Your turn